# 02 — Evaluation harness

**Phase 2 deliverable:** `NaiveLag` scored on all three tickers, walk-forward, on a real results
table. This is the project's spine — everything later plugs into it.

The harness was built **before any model**, which is the single most important ordering decision
in the project. A faithful port of a leaking pipeline is just a faster leaking pipeline.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("stock-retrofit @", ROOT)

## Walk-forward splits

No single fixed tail split anywhere (spec R6). Upstream took the last 30 rows once, *after*
scaling the whole series.

In [ ]:
from stock_retrofit.config import EvalConfig
from stock_retrofit.data import load

eval_cfg = EvalConfig.load()
df = load("KBANK")
folds = eval_cfg.splitter().split_frame(df)

print(f"{len(folds)} folds over {len(df)} bars\n")
for f in folds:
    print(" ", f.describe(df["date"]))

## The leakage guard

Intending to fit inside folds is not enough — the upstream bug is invisible in the output, since
the numbers simply come out better than they should. So every statistic that learns anything
registers the rows it saw, and the guard raises if any of them belongs to the test block.

In [ ]:
import numpy as np
from stock_retrofit.eval.leakage import LeakageError, leakage_guard, register_fit

# Correct: the fit stops exactly at the fold boundary.
with leakage_guard(test_indices=range(900, 960), fold_index=0):
    register_fit("scaler", np.arange(0, 900))
print("fit confined to training rows: allowed")

# The upstream ordering: fit the scaler on everything, split afterwards.
try:
    with leakage_guard(test_indices=range(900, 960), fold_index=0):
        register_fit("MinMaxScaler(full series)", np.arange(0, 960))
except LeakageError as exc:
    print("fit on the full series:", exc)

`tests/test_no_leakage.py` asserts exactly this, and the guard was verified by *reintroducing*
the upstream bug into `prepare_fold` and confirming the suite goes red — then removing it.

## Why the upstream metric had to go

`calculate_accuracy = 1 − sqrt(mean(((real − predict)/real)²))` on **price levels**. On a
near-random-walk series, "tomorrow's price is today's" scores in the high nineties. This is why
the upstream README's numbers look impressive.

In [ ]:
from stock_retrofit.eval.metrics import mase, upstream_accuracy_do_not_use

close = df["close"].to_numpy()
real, lag = close[1:], close[:-1]
print(f"upstream accuracy of a pure lag on KBANK prices : {upstream_accuracy_do_not_use(real, lag):.4f}")

returns = real / lag - 1.0
print(f"MASE of the same lag on returns                  : {mase(returns, np.zeros_like(returns)):.4f}")
print("\n99%+ 'accuracy' and zero skill are the same forecast, scored two ways.")

## The baseline, scored on all three tickers

`NaiveLag` is registered like any other model and appears on every results table automatically
(spec R8). Its MASE is 1.0 by construction — that is the line every other model is measured
against. It abstains from directional calls, so its accuracy is undefined rather than 0%.

In [ ]:
from stock_retrofit.config import MarketConfigSpec
from stock_retrofit.eval import render_table, results_table, run_walk_forward
from stock_retrofit.models import build

market = MarketConfigSpec.load()
results = []
for symbol in ["KBANK", "SCB", "BAY"]:
    frame = load(symbol)
    for kind in ["naive_lag", "drift", "momentum"]:
        results.append(run_walk_forward(
            build(kind, name=kind), frame,
            splitter=eval_cfg.splitter(), window=eval_cfg.window(),
            symbol=symbol, seed=eval_cfg.seed,
            cost_per_turn=market.round_trip_cost,
        ))

for symbol in ["KBANK", "SCB", "BAY"]:
    subset = [r for r in results if r.symbol == symbol]
    print(render_table(results_table(subset, cost_per_turn=market.round_trip_cost),
                       title=f"{symbol} — baselines"))
    print()